---
layout: post
toc: true
title: Impact of Program Design
menu: nav/csa_units/csaunit3.html
permalink: /csa/unit_03/3_2
---

# Impact of Program Design

A class with the right idea and the wrong design can still cause real damage. This lesson looks at what separates a well designed class from a poorly designed one, and why that difference shows up as actual bugs, not just messy looking code.

<div class="callout callout-objectives">
<h4>Objectives</h4>

- Explain how exposing raw fields lets a class end up in an invalid state
- Use encapsulation (private fields, public methods) to make invalid states hard to reach
- Distinguish cohesion (a class doing one clear job) from coupling (classes depending on each other's internals)
- Connect poor class design to concrete, real-world consequences

</div>

<style>
.lesson-wrap { font-family: 'Segoe UI', system-ui, -apple-system, sans-serif; color:#1e293b; line-height:1.65; }
.lesson-wrap h2 { color:#0f172a; border-bottom:2px solid #e2e8f0; padding-bottom:6px; margin-top:1.8em; }
.lesson-wrap h3 { color:#1e293b; margin-top:1.4em; }
.callout { border-radius:10px; padding:16px 20px; margin:16px 0; border:1px solid #e2e8f0; background:#f8fafc; }
.callout-objectives { background:#eff6ff; border-left:4px solid #2563eb; }
.callout-objectives h4 { color:#1d4ed8; margin:0 0 8px 0; }
.callout-hint { background:#fefce8; border-left:4px solid #ca8a04; }
.callout-hint h4 { color:#a16207; margin:0 0 8px 0; }
.callout-practice { background:#f0fdf4; border-left:4px solid #16a34a; }
.callout-practice h4 { color:#15803d; margin:0 0 8px 0; }
.callout-homework { background:#fdf4ff; border-left:4px solid #9333ea; }
.callout-homework h4 { color:#7e22ce; margin:0 0 8px 0; }
.callout-note { background:#f1f5f9; border-left:4px solid #64748b; }
.callout-note h4 { color:#334155; margin:0 0 8px 0; }
table { border-collapse:collapse; width:100%; margin:16px 0; }
th, td { border:1px solid #e2e8f0; padding:10px 14px; text-align:left; }
th { background:#f1f5f9; color:#0f172a; }
details { border:1px solid #e2e8f0; border-radius:8px; padding:10px 16px; margin:12px 0; background:#fafafa; }
details summary { cursor:pointer; font-weight:600; color:#2563eb; }
details[open] summary { margin-bottom:10px; }
.btn { display:inline-block; padding:10px 22px; border-radius:8px; border:none; font-weight:600; cursor:pointer; font-size:14px; transition:background .2s,transform .2s; }
.btn-primary { background:#2563eb; color:#fff; }
.btn-primary:hover { background:#1d4ed8; }
.btn-secondary { background:#e2e8f0; color:#1e293b; }
.btn-secondary:hover { background:#cbd5e1; }
.pill { display:inline-block; padding:3px 12px; border-radius:999px; background:#eff6ff; color:#1d4ed8; font-size:0.85em; font-weight:600; margin-right:6px; }
</style>

## When Design Goes Wrong

Here is a `BankAccount` class that looks reasonable at a glance. It has a balance, and you can deposit or withdraw.

In [ ]:
public class BankAccount {
    public double balance; // public field: anyone can read or write it directly

    public BankAccount(double balance) {
        this.balance = balance;
    }
}

BankAccount acct = new BankAccount(100.0);
acct.balance = acct.balance - 500; // no withdraw() method was called, no check happened
System.out.println("Balance: " + acct.balance);
// Balance: -400.0 -- a real account should never be allowed to go this negative
// this easily, but nothing in the class stopped it.

The bug here is not a typo. The class was designed to let anything reach in and change `balance` directly, with no rule enforced anywhere. `balance` being `public` was the design mistake, and no amount of careful code elsewhere fixes it, because any single line anywhere in the program can undo that care.

## Encapsulation: Designing So Mistakes Can't Happen

Encapsulation means hiding a class's fields and only allowing changes through methods that can enforce rules. The fix is not "be more careful," it's changing the design so the invalid state is unreachable.

```java
public class BankAccount {
    private double balance; // no longer reachable from outside directly

    public BankAccount(double balance) {
        this.balance = balance;
    }

    public double getBalance() {
        return balance;
    }

    public void withdraw(double amount) {
        if (amount > balance) {
            System.out.println("Withdrawal denied: insufficient funds.");
            return;
        }
        balance -= amount;
    }

    public void deposit(double amount) {
        if (amount < 0) {
            System.out.println("Deposit denied: amount cannot be negative.");
            return;
        }
        balance += amount;
    }
}
```

Now `acct.balance = -400` will not even compile from outside the class, since `balance` is `private`. Every path that changes the balance runs through a method that can check its work first.

## Cohesion and Coupling

Two more design ideas show up constantly once your programs grow past a single class:

- **Cohesion**: a class should do one clearly defined job. A `BankAccount` that also formats receipts, sends emails, and logs analytics has low cohesion. It's harder to read, harder to test, and harder to change without breaking something unrelated.
- **Coupling**: how much one class depends on another class's internal details. If a `Bank` class reaches directly into `BankAccount`'s fields instead of calling its methods, the two classes are tightly coupled. Change one field name in `BankAccount` and `Bank` breaks too. Tight coupling is exactly what encapsulation protects against, since `Bank` can only ever go through `BankAccount`'s public interface.

## Why This Is More Than Style

This is not just about writing code that looks clean. A `BankAccount` with a public balance field is a real bug waiting to happen: one careless line anywhere in a large codebase and a customer's balance is wrong. The same shape of mistake shows up constantly in real software: a shopping cart that lets item quantity go negative and refunds money it never charged, a medical dosage class that exposes a raw number instead of a validated method and lets a decimal point slip through. Good class design is not decoration. It is the thing standing between a mistake somewhere in a program and that mistake actually reaching a user.

<div class="callout callout-practice">
<h4>Popcorn Hack</h4>

Here's a leaky `Ticket` class. Rewrite it with proper encapsulation: make `price` and `quantity` private, add a constructor, and add a `getTotalCost()` method. Make sure `quantity` can never be set below 0.

```java
public class Ticket {
    public double price;
    public int quantity;
}
```

</div>

<div class="callout callout-homework">
<h4>Homework Hack</h4>

Find (or write) a small class with at least one public field that should not be public. In a markdown cell, explain:
- What invalid state that public field allows
- How you would rewrite the class using private fields and methods to prevent it
- Whether the class has more than one job (cohesion), and if so, how you would split it

</div>

Next, we'll open up a full class from top to bottom, in Anatomy of a Class, and see these design ideas expressed in real Java syntax.